# Importing Libraries

In [ ]:
import numpy as np
import pandas as pd

# Text preprocessing
import re # Remove symbols/numbers
import string # Remove punctuations 
import nltk # remove common  words
from nltk.corpus import stopwords # Stopwords are very common words that usually do not carry important meaning for classification tasks.
from nltk.stem import WordNetLemmatizer # to convert words into their root form so that different grammatical forms of the same word are treated as a single feature

#Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer # compare how important a word is in a ticket
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn import metrics
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
nltk.download('stopwords')
nltk.download('wordnet')

# Loading the dataset

In [ ]:
df = pd.read_csv("Customer IT Support.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

# Identifying null values

In [ ]:
df.isnull().sum()

In [ ]:
df = df[df['language'] == 'en'] # to keep the language of the dataset in english as the dataset is in multiple language

# Text Cleaning

In [ ]:
df["Text"] = df["subject"].fillna("") + " " + df["body"].fillna("")

In [ ]:
lemmatizer = WordNetLemmatizer()

stop_words = set(stopwords.words("english"))

stop_words.discard('not')
stop_words.discard('no') # To not remove "not" and "no" words

In [ ]:
def clean_text(text):

    text = str(text).lower()

    text = re.sub(r'[^a-zA-Z\s]','',text)

    words = text.split()

    words = [lemmatizer.lemmatize(word)
              for word in words
              if word not in stop_words]
    return " ".join(words)

In [ ]:
df["Clean_Text"] = df["Text"].apply(clean_text)

# Exploratory Data Analysis

#### Queue Distribution

In [ ]:
plt.figure(figsize=(20,6))

sns.countplot(data=df,x='queue')

plt.title("Queue Distribution")

plt.xticks(rotation = 90)

plt.show()

#### Priority Distribution

In [ ]:
plt.figure(figsize = (15,6))

df["priority"].value_counts().sort_values(ascending = True).plot( kind='pie', autopct='%1.1f%%')

plt.title("Priority Distribution")

plt.show()

# Coverting Text into Numbers

* TF = Term Frequency ( Counts how often a word appears )
* IDF = Inverse Document Frequency ( Words appearing in many tickets get lower importance )

In [ ]:
# uses ngram_range so that model can learn both individual words and two-word phrases

# uses min_df to remove extremel rare words

vectorizer = TfidfVectorizer( max_features = 10000, ngram_range=(1,2), min_df=2) # keep only the 10000 most important words from the entire dataset

# Category Classification

In [ ]:
y_category = df["queue"]

# Train Test Split

In [ ]:
X_train_text, X_test_text, y_train_cat, y_test_cat = train_test_split(df["Clean_Text"], y_category, test_size = 0.2, random_state = 42, stratify=y_category)

X_train_cat = vectorizer.fit_transform(X_train_text)

X_test_cat = vectorizer.transform(X_test_text)

# Logistic Regression

In [ ]:
category_model = LogisticRegression(max_iter = 1000) # To ensure that the model reaches an optimal solution without convergence warnings

category_model.fit(X_train_cat, y_train_cat)

In [ ]:
y_pred_cat = category_model.predict(X_test_cat)

In [ ]:
print("Accuracy:", accuracy_score(y_test_cat,y_pred_cat))

In [ ]:
print(classification_report(y_test_cat,y_pred_cat))

In [ ]:
cm = confusion_matrix(y_test_cat,y_pred_cat)

plt.figure(figsize=(20,8))

sns.heatmap(cm,annot=True,fmt='d')

plt.show()

# Priority Prediction

In [ ]:
y_priority = df["priority"]

# Train Test Split

In [ ]:
X_train, X_test, y_train_pri, y_test_pri = train_test_split(df['Clean_Text'],y_priority,test_size=0.2,random_state=42,stratify=y_priority)

# TF - IDF

In [ ]:
X_train_pri = vectorizer.fit_transform(X_train)

X_test_pri = vectorizer.transform(X_test)

In [ ]:
priority_model = LogisticRegression( max_iter=1000)

priority_model.fit(X_train_pri,y_train_pri)

In [ ]:
priority_pred = priority_model.predict(X_test_pri)

print("Accuracy",accuracy_score(y_test_pri,priority_pred))

In [ ]:
print(classification_report(y_test_pri,priority_pred))

# Support Analysis

#### Top Queues

In [ ]:
df["queue"].value_counts()

#### Top Priority

In [ ]:
df["priority"].value_counts()

# Saving the models

In [ ]:
import joblib

joblib.dump(category_model,"category_model.pkl")

joblib.dump(priority_model,"priority_model.pkl")

joblib.dump(vectorizer,"vectorizer.pkl")

In [ ]:
def predict_ticket(ticket):

    cleaned = clean_text(ticket)

    vector = vectorizer.transform([cleaned])

    category = category_model.predict(vector)[0]

    priority = priority_model.predict(vector)[0]

    return category, priority